In [0]:
# =====================================================
# 1. DEFINE ADF WIDGET PARAMETERS
# =====================================================

dbutils.widgets.text("source_system", "", "")
dbutils.widgets.text("table_name", "", "")
dbutils.widgets.text("landing_timestamp", "", "")
dbutils.widgets.text("source_file_format", "", "")

# Get parameters from ADF
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


In [0]:
# =====================================================
# 2. BUILD LANDING PATH AND CHECK IF IT EXISTS
# =====================================================

landing_path = f"/Volumes/adb_carsync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        if "FileNotFound" in str(e) or "does not exist" in str(e).lower():
            return False
        raise

if not path_exists(landing_path):
    print(f"No new files received. Path does not exist: {landing_path}")
    dbutils.notebook.exit(f"No new files received for {source_system}/{table_name}/{landing_timestamp}")

print(f"Landing path found: {landing_path}")

In [0]:

print("\n========== READING LANDING DATA ==========")

if source_file_format == "csv":
     landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

    
print("\nLanding data loaded successfully!")


In [0]:
from pyspark.sql import functions as F
updated_df = (
    landing_df
    .withColumn(
        "landing_timestamp",
        F.lit(landing_timestamp)
    )
    .withColumn(
        "insert_timestamp",
        F.current_timestamp()
    )
)

print("\nAudit columns added successfully!")

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_carsync.bronze.{table_name}")
print(f"\nData written to table: adb_carsync.bronze.{table_name}")